# Hybrid neural density estimation

Reproduce Figures 1, 2, 3(a), 4, 8 and 10 of the paper.

In [ ]:
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/hnsbi_asimov")
    repo = project / "repository"
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/rafaellopesdesa/hnsbi_asimov.git", str(repo)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")],
        check=True,
    )
else:
    repo = Path.cwd()
    project = repo
sys.path.insert(0, str(repo))
from utils import setup_workspace

setup_workspace(repo, project / "workspace")

## Generate $p_S(\mathbf{x})$ and $p_B(\mathbf{x})$

In [ ]:
if not Path("dataframes/signal.parquet").exists():
    subprocess.run([sys.executable, str(repo / "generate_distributions.py")], check=True)

In [ ]:
import gc
import jax

jax.config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp
from scipy.stats import multivariate_normal
from utils import (
    FEATURES,
    as_inference_session,
    density_ratio_trainer,
    evaluate_ratio_packs,
    predict_with_model,
    ratio_training_dataframe,
    sample_selected_flow,
)
from utils_distributions import background_components, signal_components, smearing_parameters
from utils_nf import (
    accumulate_preselection_histogram,
    choose_preselection_ratio_cut,
    collect_preselected_parquet,
    flow_log_prob_x,
    sample_parquet_partition,
    train_flow,
)
from utils_plotting import (
    plot_flow_pair_closure,
    plot_log_density_truth_binned,
    plot_toy_comparison,
)
from utils_toys import (
    build_compressed_q_model,
    make_toy_fitter,
    run_toys,
    load_or_generate_simulator_q_bank,
    probability_in_log_q_bins,
)

SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_DIM = len(FEATURES)
BASE_PATH = Path("./dataframes")
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}
for directory in [PRESEL_MODEL_DIR, REFERENCE_FLOW_MODEL_DIR, *RATIO_MODEL_DIR.values()]:
    directory.mkdir(parents=True, exist_ok=True)
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.92
STREAM_BATCH_SIZE = 100000
PRESEL_CUT_HISTOGRAM_BINS = 4000
PRESEL_LOG_RATIO_RANGE = (-20.0, 20.0)
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 250.0
PRESEL_MAX_TRAIN_EVENTS_PER_CLASS = 500000
PRESEL_HIDDEN_LAYERS = 3
PRESEL_NEURONS = 256
PRESEL_N_EPOCHS = 30
PRESEL_BATCH_SIZE = 2048
PRESEL_LEARNING_RATE = 0.001
PRESEL_HOLDOUT_FRACTION = 0.25
PRESEL_VALIDATION_FRACTION = 0.2
PRESEL_PATIENCE = 8
PRESEL_LOAD_IF_AVAILABLE = True
MAX_TRAIN_EVENTS = {"background": 5000000, "signal": 5000000}
MAX_EVAL_EVENTS = {"background": 250000, "signal": 250000}
REFERENCE_COMPONENT_TRAIN_EVENTS = 500000
N_REFERENCE_EVENTS = 5000000
REFERENCE_SAMPLING_BATCH_SIZE = 65536
FLOW_TYPE = "quadratic_spline"
N_COUPLING_LAYERS = 10
HIDDEN_FEATURES = 1024
HIDDEN_LAYERS = 4
SPLINE_NUM_BINS = 16
SPLINE_TAIL_BOUND = 5.0
DROPOUT_PROBABILITY = 0.0
BATCH_SIZE = 2048
N_EPOCHS = 70
LEARNING_RATE = 0.0001
LR_SCHEDULER_FACTOR = 0.2
LR_SCHEDULER_PATIENCE = 2
MIN_LEARNING_RATE = 1e-07
WEIGHT_DECAY = 0.0
VALIDATION_FRACTION = 0.2
PATIENCE = 5
GRADIENT_CLIP = 5.0
FLOW_LOAD_IF_AVAILABLE = True
MODEL_CONFIG = {
    "flow_type": FLOW_TYPE,
    "n_features": N_DIM,
    "n_coupling_layers": N_COUPLING_LAYERS,
    "hidden_features": HIDDEN_FEATURES,
    "hidden_layers": HIDDEN_LAYERS,
    "spline_num_bins": SPLINE_NUM_BINS,
    "spline_tail_bound": SPLINE_TAIL_BOUND,
    "dropout_probability": DROPOUT_PROBABILITY,
}
TRAINING_CONFIG = {
    "batch_size": BATCH_SIZE,
    "n_epochs": N_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "lr_scheduler_factor": LR_SCHEDULER_FACTOR,
    "lr_scheduler_patience": LR_SCHEDULER_PATIENCE,
    "min_learning_rate": MIN_LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "validation_fraction": VALIDATION_FRACTION,
    "patience": PATIENCE,
    "gradient_clip": GRADIENT_CLIP,
}
MAX_RATIO_EVENTS_PER_CLASS = 5000000
RATIO_ENSEMBLE_SIZE = 4
RATIO_HIDDEN_LAYERS = 4
RATIO_NEURONS = 1024
RATIO_N_EPOCHS = 50
RATIO_BATCH_SIZE = 4096
RATIO_LEARNING_RATE = 0.001
RATIO_HOLDOUT_FRACTION = 0.25
RATIO_VALIDATION_FRACTION = 0.2
RATIO_PATIENCE = 10
RATIO_LOAD_IF_AVAILABLE = True
RATIO_EVALUATION_BATCH_SIZE = 100000

## Preselection and independent training and evaluation samples

In [ ]:
SAMPLE_PATHS = {
    "signal": BASE_PATH / "signal.parquet",
    "background": BASE_PATH / "background.parquet",
}
PRESEL_signal_bce, PRESEL_signal_input_stats = sample_parquet_partition(
    SAMPLE_PATHS["signal"],
    features=FEATURES,
    partition="presel",
    max_events=PRESEL_MAX_TRAIN_EVENTS_PER_CLASS,
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
    reservoir_seed=SEED + 11,
)
PRESEL_background_bce, PRESEL_background_input_stats = sample_parquet_partition(
    SAMPLE_PATHS["background"],
    features=FEATURES,
    partition="presel",
    max_events=PRESEL_MAX_TRAIN_EVENTS_PER_CLASS,
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
    reservoir_seed=SEED + 22,
)
PRESEL_INPUT_STATS = {
    "signal": PRESEL_signal_input_stats,
    "background": PRESEL_background_input_stats,
}
PRESEL_INCLUSIVE_YIELD = {
    sample_name: float(stats["inclusive_weight"])
    for sample_name, stats in PRESEL_INPUT_STATS.items()
}

In [ ]:
for PRESEL_sample, PRESEL_label in [(PRESEL_signal_bce, 1.0), (PRESEL_background_bce, 0.0)]:
    PRESEL_sample["PRESEL_label"] = PRESEL_label
    PRESEL_sample["PRESEL_weight"] = PRESEL_sample["weight"] / PRESEL_sample["weight"].sum()
PRESEL_training_dataframe = pd.concat(
    [PRESEL_signal_bce, PRESEL_background_bce], ignore_index=True
).sample(frac=1.0, random_state=SEED, ignore_index=True)
PRESEL_trainer = density_ratio_trainer(
    dataset=PRESEL_training_dataframe,
    weights=PRESEL_training_dataframe["PRESEL_weight"].to_numpy(),
    training_labels=PRESEL_training_dataframe["PRESEL_label"].to_numpy(),
    features=FEATURES,
    features_scaling=FEATURES,
    sample_name=["signal", "background"],
    output_name="PRESEL",
    path_to_figures=f"{PRESEL_MODEL_DIR}/",
    path_to_models=f"{PRESEL_MODEL_DIR}/",
)
PRESEL_trainer.train(
    hidden_layers=PRESEL_HIDDEN_LAYERS,
    neurons=PRESEL_NEURONS,
    number_of_epochs=PRESEL_N_EPOCHS,
    batch_size=PRESEL_BATCH_SIZE,
    learning_rate=PRESEL_LEARNING_RATE,
    scalerType="MinMax",
    ensemble_index=0,
    verbose=1,
    rnd_seed=SEED,
    holdout_split=PRESEL_HOLDOUT_FRACTION,
    validation_split=PRESEL_VALIDATION_FRACTION,
    callback_patience=PRESEL_PATIENCE,
    num_workers=0,
    load_trained_models=PRESEL_LOAD_IF_AVAILABLE,
    calibration=False,
)
PRESEL_scaler = PRESEL_trainer.scaler
PRESEL_model = as_inference_session(PRESEL_trainer.model_NN)
del (
    PRESEL_trainer,
    PRESEL_training_dataframe,
    PRESEL_signal_bce,
    PRESEL_background_bce,
    PRESEL_sample,
)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
def evaluate_PRESEL_ratio(feature_dataframe):
    ratio = predict_with_model(
        feature_dataframe.astype("float32", copy=False), scaler=PRESEL_scaler, model=PRESEL_model
    )
    return np.asarray(ratio, dtype=float).reshape(-1)


PRESEL_LOG_RATIO_EDGES = np.linspace(
    PRESEL_LOG_RATIO_RANGE[0], PRESEL_LOG_RATIO_RANGE[1], PRESEL_CUT_HISTOGRAM_BINS + 1
)
PRESEL_signal_histogram, PRESEL_signal_hist_stats = accumulate_preselection_histogram(
    SAMPLE_PATHS["signal"],
    features=FEATURES,
    ratio_predictor=evaluate_PRESEL_ratio,
    log_ratio_edges=PRESEL_LOG_RATIO_EDGES,
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
)
PRESEL_background_histogram, PRESEL_background_hist_stats = accumulate_preselection_histogram(
    SAMPLE_PATHS["background"],
    features=FEATURES,
    ratio_predictor=evaluate_PRESEL_ratio,
    log_ratio_edges=PRESEL_LOG_RATIO_EDGES,
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
)
PRESEL_RATIO_CUT, PRESEL_CUT_DIAGNOSTICS = choose_preselection_ratio_cut(
    PRESEL_signal_histogram,
    PRESEL_background_histogram,
    PRESEL_LOG_RATIO_EDGES,
    signal_inclusive_yield=PRESEL_INCLUSIVE_YIELD["signal"],
    background_inclusive_yield=PRESEL_INCLUSIVE_YIELD["background"],
    signal_partition_weight=PRESEL_signal_hist_stats["partition_weight"],
    background_partition_weight=PRESEL_background_hist_stats["partition_weight"],
    target_background_to_signal=PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
)
PRESEL_signal_samples, PRESEL_signal_stream_stats = collect_preselected_parquet(
    SAMPLE_PATHS["signal"],
    features=FEATURES,
    ratio_predictor=evaluate_PRESEL_ratio,
    ratio_cut=PRESEL_RATIO_CUT,
    max_train_events=MAX_TRAIN_EVENTS["signal"],
    max_eval_events=MAX_EVAL_EVENTS["signal"],
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
    reservoir_seed=SEED + 101,
)
PRESEL_background_samples, PRESEL_background_stream_stats = collect_preselected_parquet(
    SAMPLE_PATHS["background"],
    features=FEATURES,
    ratio_predictor=evaluate_PRESEL_ratio,
    ratio_cut=PRESEL_RATIO_CUT,
    max_train_events=MAX_TRAIN_EVENTS["background"],
    max_eval_events=MAX_EVAL_EVENTS["background"],
    batch_size=STREAM_BATCH_SIZE,
    presel_fraction=PRESEL_TRAIN_FRACTION,
    flow_train_fraction=FLOW_TRAIN_FRACTION,
    split_seed=SPLIT_SEED,
    reservoir_seed=SEED + 202,
)
PRESEL_STREAM_STATS = {
    "signal": PRESEL_signal_stream_stats,
    "background": PRESEL_background_stream_stats,
}
PRESEL_SELECTED_YIELD = {}
PRESEL_EFFICIENCY = {}
for sample_name, stats in PRESEL_STREAM_STATS.items():
    train_stats = stats["flow_train"]
    PRESEL_EFFICIENCY[sample_name] = (
        train_stats["selected_weight"] / train_stats["partition_weight"]
    )
    PRESEL_SELECTED_YIELD[sample_name] = (
        PRESEL_INCLUSIVE_YIELD[sample_name] * PRESEL_EFFICIENCY[sample_name]
    )
signal_train = PRESEL_signal_samples["flow_train"]
signal_eval = PRESEL_signal_samples["eval"]
background_train = PRESEL_background_samples["flow_train"]
background_eval = PRESEL_background_samples["eval"]
for sample_name, train_sample, eval_sample in [
    ("signal", signal_train, signal_eval),
    ("background", background_train, background_eval),
]:
    for split_name, sample in [("train", train_sample), ("eval", eval_sample)]:
        retained_weight = float(sample["weight"].sum())
        sample["weight"] *= PRESEL_SELECTED_YIELD[sample_name] / retained_weight
TOTAL_YIELD = PRESEL_SELECTED_YIELD.copy()
del (
    PRESEL_signal_samples,
    PRESEL_background_samples,
    PRESEL_signal_histogram,
    PRESEL_background_histogram,
)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
np.savez(
    PRESEL_MODEL_DIR / "selection.npz",
    ratio_cut=PRESEL_RATIO_CUT,
    lambda_signal=TOTAL_YIELD["signal"],
    lambda_background=TOTAL_YIELD["background"],
    inclusive_signal=PRESEL_INCLUSIVE_YIELD["signal"],
    inclusive_background=PRESEL_INCLUSIVE_YIELD["background"],
    nis_lambda_signal=PRESEL_CUT_DIAGNOSTICS["histogram_signal_yield"],
    nis_lambda_background=PRESEL_CUT_DIAGNOSTICS["histogram_background_yield"],
)

## Train $q_{\boldsymbol{\phi}}(\mathbf{x})$ (Algorithm 1)

In [ ]:
def make_balanced_reference(signal_df, background_df, n_per_component, seed):
    signal_part = signal_df.sample(n=n_per_component, random_state=seed)[FEATURES].copy()
    background_part = background_df.sample(n=n_per_component, random_state=seed + 1)[
        FEATURES
    ].copy()
    signal_part["weight"] = 0.5 / n_per_component
    background_part["weight"] = 0.5 / n_per_component
    return pd.concat([signal_part, background_part], ignore_index=True).sample(
        frac=1.0, random_state=seed + 2, ignore_index=True
    )


reference_flow_train = make_balanced_reference(
    signal_train, background_train, REFERENCE_COMPONENT_TRAIN_EVENTS, SEED + 301
)
reference_flow = train_flow(
    "reference",
    reference_flow_train,
    features=FEATURES,
    model_dir=REFERENCE_FLOW_MODEL_DIR,
    model_config=MODEL_CONFIG,
    training_config=TRAINING_CONFIG,
    device=device,
    load_if_available=FLOW_LOAD_IF_AVAILABLE,
    seed=SEED + 501,
)
del reference_flow_train
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Draw $\mathbf{x}_m\sim q_{\boldsymbol{\phi}}$

In [ ]:
def sample_preselected_reference_flow(flow, n_events, batch_size=65536):
    return sample_selected_flow(
        flow, n_events, lambda x: evaluate_PRESEL_ratio(x) >= PRESEL_RATIO_CUT, batch_size
    )


reference_values, REFERENCE_FLOW_PRESEL_ACCEPTANCE = sample_preselected_reference_flow(
    reference_flow, N_REFERENCE_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
reference_sample = pd.DataFrame(reference_values, columns=FEATURES)
reference_sample["weight"] = 1.0 / len(reference_sample)
del reference_values


def reference_log_prob_x(x, batch_size=65536):
    return np.asarray(
        flow_log_prob_x(reference_flow, x, batch_size=batch_size), dtype=np.float64
    ) - np.log(REFERENCE_FLOW_PRESEL_ACCEPTANCE)

## Train $r_{s,\boldsymbol{\psi}}(\mathbf{x})$, $s\in\{S,B\}$ (Algorithm 1)

In [ ]:
def train_ratio_ensemble(sample_name, target, seed):
    data = ratio_training_dataframe(target, reference_sample, MAX_RATIO_EVENTS_PER_CLASS, seed)
    packs = []
    for member in range(RATIO_ENSEMBLE_SIZE):
        np.random.seed(seed + 10000 + member)
        torch.manual_seed(seed + 10000 + member)
        trainer = density_ratio_trainer(
            dataset=data,
            weights=data["weights_normed"],
            training_labels=data["train_labels"],
            features=FEATURES,
            features_scaling=FEATURES,
            sample_name=[sample_name, "reference"],
            output_name="",
            path_to_figures=f"{RATIO_MODEL_DIR[sample_name]}/",
            path_to_models=f"{RATIO_MODEL_DIR[sample_name]}/",
        )
        trainer.train(
            hidden_layers=RATIO_HIDDEN_LAYERS,
            neurons=RATIO_NEURONS,
            number_of_epochs=RATIO_N_EPOCHS,
            batch_size=RATIO_BATCH_SIZE,
            learning_rate=RATIO_LEARNING_RATE,
            scalerType="MinMax",
            ensemble_index=member,
            verbose=1,
            rnd_seed=seed,
            holdout_split=RATIO_HOLDOUT_FRACTION,
            validation_split=RATIO_VALIDATION_FRACTION,
            callback_patience=RATIO_PATIENCE,
            num_workers=0,
            load_trained_models=RATIO_LOAD_IF_AVAILABLE,
            calibration=False,
        )
        packs.append({"scaler": trainer.scaler, "model": as_inference_session(trainer.model_NN)})
        del trainer
        gc.collect()
        torch.cuda.empty_cache()
    return packs


ratio_models = {}
ratio_models["signal"] = train_ratio_ensemble("signal", signal_train, SEED + 601)
del signal_train
gc.collect()
ratio_models["background"] = train_ratio_ensemble("background", background_train, SEED + 701)
del background_train
gc.collect()

## Normalize $\widetilde r_{s,\boldsymbol{\psi}}$ on $\mathcal X_M$ (Section 4.2)

In [ ]:
def evaluate_ratio(sample_name, dataframe, batch_size=100000):
    return evaluate_ratio_packs(ratio_models[sample_name], dataframe[FEATURES], batch_size)


RATIO_NORMALIZATION = {}
for sample_name in ["signal", "background"]:
    raw_ratio = evaluate_ratio(
        sample_name, reference_sample, batch_size=RATIO_EVALUATION_BATCH_SIZE
    )
    RATIO_NORMALIZATION[sample_name] = float(raw_ratio.mean())
    normalized_ratio = raw_ratio / RATIO_NORMALIZATION[sample_name]
    reference_sample[f"ratio_{sample_name}"] = normalized_ratio
    reference_sample[f"weight_{sample_name}"] = normalized_ratio / len(reference_sample)

## Figure 1: background reweighting with $r_{B,\boldsymbol{\psi}}$

In [ ]:
reference_plot = reference_sample.sample(n=100000, random_state=SEED + 820)
background_plot = background_eval.sample(n=100000, random_state=SEED + 821)
plot_flow_pair_closure(
    "background",
    background_plot[FEATURES].to_numpy(),
    reference_plot[FEATURES].to_numpy(),
    FEATURES,
    mc_weights=background_plot["weight"].to_numpy(),
    generated_weights=reference_plot["weight_background"].to_numpy(),
    mc_label="Background",
    generated_label="Weighted reference",
    generated_color="C0",
    correlation_names=("target", "w.ref."),
)
plt.show()
del reference_plot, background_plot

## Figure 2: $\log\widehat p_S(\mathbf{x})$ and $\log p_S(\mathbf{x})$

In [ ]:
def reco_components_from_truth_components(components):
    scale, resolution = smearing_parameters()
    scale = np.asarray(scale, dtype=float)
    resolution = np.asarray(resolution, dtype=float)
    D = np.diag(scale)
    response_cov = np.diag(resolution**2)
    reco_components = []
    for frac, mean_y, cov_y in components:
        mean_x = scale * np.asarray(mean_y, dtype=float)
        cov_x = D @ np.asarray(cov_y, dtype=float) @ D.T + response_cov
        reco_components.append((frac, mean_x, cov_x))
    return reco_components


def mixture_log_density(x, components):
    x = np.asarray(x, dtype=float)
    fracs = np.asarray([component[0] for component in components], dtype=float)
    fracs = fracs / fracs.sum()
    terms = [
        np.log(frac) + multivariate_normal(mean=mean, cov=cov, allow_singular=False).logpdf(x)
        for frac, (_, mean, cov) in zip(fracs, components)
    ]
    return logsumexp(np.vstack(terms), axis=0)


TRUTH_RECO_COMPONENTS = {
    "background": reco_components_from_truth_components(background_components()),
    "signal": reco_components_from_truth_components(signal_components()),
}


def selected_truth_log_density(x, sample_name):
    return mixture_log_density(x, TRUTH_RECO_COMPONENTS[sample_name]) - np.log(
        PRESEL_EFFICIENCY[sample_name]
    )


asimov_dataset = pd.concat([background_eval, signal_eval], ignore_index=True).copy()
weights_asimov = asimov_dataset["weight"].to_numpy(dtype=np.float64)
X_asimov = asimov_dataset[FEATURES].to_numpy(dtype=np.float32)
lam_sig = float(TOTAL_YIELD["signal"])
lam_bkg = float(TOTAL_YIELD["background"])
log_p_ref_asimov = reference_log_prob_x(X_asimov)
HYBRID_RATIOS = {}
HYBRID_LOG_DENSITIES = {}
for sample_name in ["signal", "background"]:
    ratio = (
        evaluate_ratio(sample_name, asimov_dataset, batch_size=RATIO_EVALUATION_BATCH_SIZE)
        / RATIO_NORMALIZATION[sample_name]
    )
    HYBRID_RATIOS[sample_name] = ratio
    HYBRID_LOG_DENSITIES[sample_name] = log_p_ref_asimov + np.log(ratio)
TRUTH_LOG_DENSITIES = {
    "signal": selected_truth_log_density(X_asimov, "signal"),
    "background": selected_truth_log_density(X_asimov, "background"),
}
plot_log_density_truth_binned(TRUTH_LOG_DENSITIES["signal"], HYBRID_LOG_DENSITIES["signal"])
plt.show()

## Figure 3(a): $t_\mu$ from $\widehat p_s$ and $p_s$

In [ ]:
def density_scan(log_signal, log_background, mu_values):
    log_q = np.log(lam_sig / lam_bkg) + log_signal - log_background
    q = np.exp(log_q)

    def nll(mu):
        return 2 * (mu * lam_sig - np.sum(weights_asimov * np.log1p(mu * q)))

    minimum = minimize_scalar(nll, bounds=(0, 3), method="bounded")
    return np.array([nll(mu) - minimum.fun for mu in mu_values])


scan_mu = np.linspace(0, 3, 50)
tmu_hybrid = density_scan(
    HYBRID_LOG_DENSITIES["signal"], HYBRID_LOG_DENSITIES["background"], scan_mu
)
tmu_truth = density_scan(TRUTH_LOG_DENSITIES["signal"], TRUTH_LOG_DENSITIES["background"], scan_mu)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(scan_mu, tmu_hybrid, label="hNDE")
ax.plot(scan_mu, tmu_truth, ls="--", label="Analytic density")
ax.axvline(1, color="black", ls=":")
ax.set(xlabel="$\\mu$", ylabel="$t_\\mu$", ylim=(0, 5))
ax.legend()
plt.show()

## Construct $\mathcal A_M(\boldsymbol{\theta}_A)$ and $q_{0,A}$ (Algorithm 2)

In [ ]:
ASIMOV_MU_TRUE = 1.0
reference_ratio_signal = reference_sample["ratio_signal"].to_numpy(dtype=np.float64)
reference_ratio_background = reference_sample["ratio_background"].to_numpy(dtype=np.float64)
reference_weight_signal = reference_sample["weight_signal"].to_numpy(dtype=np.float64)
reference_weight_background = reference_sample["weight_background"].to_numpy(dtype=np.float64)
REFERENCE_LOG_Q = (
    np.log(lam_sig / lam_bkg) + np.log(reference_ratio_signal) - np.log(reference_ratio_background)
)
REFERENCE_Q = np.exp(np.clip(REFERENCE_LOG_Q, -80.0, 80.0))
WEIGHTED_ASIMOV_WEIGHTS = (
    ASIMOV_MU_TRUE * lam_sig * reference_weight_signal + lam_bkg * reference_weight_background
)
ASIMOV_Q_ZERO = 2 * (
    -ASIMOV_MU_TRUE * lam_sig
    + np.sum(WEIGHTED_ASIMOV_WEIGHTS * np.log1p(ASIMOV_MU_TRUE * REFERENCE_Q))
)
ASIMOV_SIGMA_MU = ASIMOV_MU_TRUE / np.sqrt(ASIMOV_Q_ZERO)

## Figure 4: $\widehat\mu$ and $q_0$ from $\widehat P$ and the Asimov prediction

In [ ]:
N_TOYS = 100000
TOY_BATCH_SIZE = 2000
TOY_Q_BINS = 512
COMPRESSED_TOY_MODEL = build_compressed_q_model(
    REFERENCE_Q, reference_weight_signal, reference_weight_background, lam_sig, lam_bkg, TOY_Q_BINS
)
fit_toys = make_toy_fitter(COMPRESSED_TOY_MODEL["q"], lam_sig, lam_bkg)
hybrid_toy_results = run_toys(
    ASIMOV_MU_TRUE,
    N_TOYS,
    315159,
    COMPRESSED_TOY_MODEL["signal_probability"],
    COMPRESSED_TOY_MODEL["background_probability"],
    lam_sig,
    lam_bkg,
    TOY_BATCH_SIZE,
    fit_toys,
)
plot_toy_comparison(hybrid_toy_results, ASIMOV_MU_TRUE, ASIMOV_SIGMA_MU, ASIMOV_Q_ZERO)
plt.show()

## Figures 8 and 10: $\widehat\mu$ and $q_0$ from $\widehat P$ and $P_{\rm sim}$

In [ ]:
SIMULATOR_BANK_SELECTED_EVENTS = 5000000
SIMULATOR_BANK_GENERATION_BATCH_SIZE = 250000
SIMULATOR_BANK_SEED = 271828
SIMULATOR_BANK_DIR = Path("simulator_toy_banks_hybrid")
response_scale, response_resolution = smearing_parameters()
simulator_bank_options = {
    "n_selected": SIMULATOR_BANK_SELECTED_EVENTS,
    "generation_batch_size": SIMULATOR_BANK_GENERATION_BATCH_SIZE,
    "bank_dir": SIMULATOR_BANK_DIR,
    "feature_names": FEATURES,
    "response_scale": response_scale,
    "response_resolution": response_resolution,
    "presel_ratio_cut": PRESEL_RATIO_CUT,
    "evaluate_presel_ratio": evaluate_PRESEL_ratio,
    "evaluate_ratio": evaluate_ratio,
    "ratio_normalization": RATIO_NORMALIZATION,
    "lam_sig": lam_sig,
    "lam_bkg": lam_bkg,
    "ratio_evaluation_batch_size": RATIO_EVALUATION_BATCH_SIZE,
}
SIMULATOR_Q_SIGNAL = load_or_generate_simulator_q_bank(
    sample_name="signal",
    components=signal_components(),
    seed=SIMULATOR_BANK_SEED + 1,
    **simulator_bank_options,
)
SIMULATOR_Q_BACKGROUND = load_or_generate_simulator_q_bank(
    sample_name="background",
    components=background_components(),
    seed=SIMULATOR_BANK_SEED + 2,
    **simulator_bank_options,
)
SIMULATOR_SIGNAL_PROBABILITY = probability_in_log_q_bins(
    SIMULATOR_Q_SIGNAL, COMPRESSED_TOY_MODEL["log_q_edges"]
)
SIMULATOR_BACKGROUND_PROBABILITY = probability_in_log_q_bins(
    SIMULATOR_Q_BACKGROUND, COMPRESSED_TOY_MODEL["log_q_edges"]
)
simulator_toy_results = run_toys(
    ASIMOV_MU_TRUE,
    N_TOYS,
    172803,
    SIMULATOR_SIGNAL_PROBABILITY,
    SIMULATOR_BACKGROUND_PROBABILITY,
    lam_sig,
    lam_bkg,
    TOY_BATCH_SIZE,
    fit_toys,
)
plot_toy_comparison(
    hybrid_toy_results,
    ASIMOV_MU_TRUE,
    ASIMOV_SIGMA_MU,
    ASIMOV_Q_ZERO,
    simulator_toys=simulator_toy_results,
    asymptotic=False,
)
plt.show()
plot_toy_comparison(
    hybrid_toy_results,
    ASIMOV_MU_TRUE,
    ASIMOV_SIGMA_MU,
    ASIMOV_Q_ZERO,
    simulator_toys=simulator_toy_results,
    log_q=True,
)
plt.show()